# 03 - Gold : indicateurs pour le dashboard
**Objectif :** créer des tables prêtes à afficher, à partir des tables Silver.

- **`station_latest`** : dernier état connu de chaque station (pour une carte)
- **`station_hourly`** : remplissage moyen par station et par heure (semaine / week-end)
- **`network_summary`** : vue globale du réseau à chaque relevé

Les tables Gold sont **entièrement recalculées** à chaque exécution (`CREATE OR REPLACE`) : elles sont petites et dépendent de toutes les données Silver.

## 1. Dernier état de chaque station
Pour chaque station : le relevé le plus récent, le taux de remplissage, les bornes indisponibles et un statut (vide / pleine / normal).

In [0]:
%sql
-- =====================================================================
-- Dernier état connu de chaque station, avec ses infos (nom, GPS)
-- Table utilisée pour la carte du dashboard
-- =====================================================================
CREATE OR REPLACE TABLE workspace.velib_gold.station_latest AS
SELECT
  s.station_id,                          -- identifiant de la station
  s.name,                                -- nom de la station
  s.latitude,                            -- position GPS
  s.longitude,
  s.capacity,                            -- nombre total de bornes
  r.snapshot_time,                       -- heure du relevé (Paris)
  r.last_reported_time,                  -- dernier signal de la station (Paris)

  -- Âge de la donnée en heures = temps entre le dernier signal et le relevé
  ROUND((unix_timestamp(r.snapshot_time) - unix_timestamp(r.last_reported_time)) / 3600, 1) AS data_age_hours,

  r.mechanical_bikes,                    -- vélos mécaniques dispos
  r.ebikes,                              -- vélos électriques dispos
  r.bikes_available,                     -- total vélos dispos
  r.docks_available,                     -- places libres pour rendre un vélo

  -- Bornes indisponibles = capacité - vélos - places libres (bornes en panne, vélos bloqués)
  s.capacity - r.bikes_available - r.docks_available AS unavailable_docks,

  -- Taux de remplissage en % (NULLIF évite une division par 0 si capacité = 0)
  ROUND(100 * r.bikes_available / NULLIF(s.capacity, 0), 1) AS fill_rate_pct,

  -- Statut lisible pour le dashboard (l'ordre compte : le 1er cas vrai gagne)
  CASE
    -- Pas de signal depuis plus de 3 h : les chiffres ne sont pas fiables
    WHEN r.last_reported_time < r.snapshot_time - INTERVAL 3 HOURS THEN 'déconnectée'
    WHEN NOT r.is_renting AND NOT r.is_returning THEN 'hors service'  -- station fermée
    WHEN r.bikes_available = 0 THEN 'vide'                          -- plus aucun vélo
    WHEN r.docks_available = 0 THEN 'pleine'                        -- plus de place pour rendre
    ELSE 'normal'
  END AS status,

  -- Grande station (plus de 100 places) : à mettre à part dans les analyses
  s.capacity > 100 AS is_large_station

FROM workspace.velib_silver.station_status AS r
JOIN workspace.velib_silver.stations       AS s ON r.station_id = s.station_id

-- On garde seulement le relevé le plus récent de chaque station
QUALIFY ROW_NUMBER() OVER (PARTITION BY r.station_id ORDER BY r.snapshot_time DESC) = 1;

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- Vérification : nombre de stations dans la table Gold
SELECT COUNT(*) AS nb_stations
FROM workspace.velib_gold.station_latest;

nb_stations
1516


## 2. Remplissage par station et par heure
Moyennes calculées sur tous les relevés, séparées entre **semaine** et **week-end** (les usages ne sont pas les mêmes).
Plus il y aura de fichiers à des heures différentes, plus cette table sera intéressante.

In [0]:
%sql
-- =====================================================================
-- Moyennes par station, par heure de la journée et par type de jour
-- On exclut les relevés de stations déconnectées (chiffres périmés)
-- =====================================================================
CREATE OR REPLACE TABLE workspace.velib_gold.station_hourly AS
SELECT
  s.station_id,
  s.name,
  HOUR(r.snapshot_time) AS hour_of_day,   -- ex : 18 pour 18h50

  -- DAYOFWEEK : 1 = dimanche, 7 = samedi
  CASE WHEN DAYOFWEEK(r.snapshot_time) IN (1, 7) THEN 'week-end' ELSE 'semaine' END AS day_type,

  COUNT(*)                          AS nb_releves,      -- nombre de relevés utilisés pour la moyenne
  ROUND(AVG(r.mechanical_bikes), 1) AS avg_mechanical,  -- moyenne de vélos mécaniques
  ROUND(AVG(r.ebikes), 1)           AS avg_ebikes,      -- moyenne de vélos électriques
  ROUND(AVG(r.docks_available), 1)  AS avg_docks,       -- moyenne de places libres

  -- % des relevés où la station était vide / pleine
  ROUND(100 * AVG(CASE WHEN r.bikes_available = 0 THEN 1 ELSE 0 END), 1) AS pct_empty,
  ROUND(100 * AVG(CASE WHEN r.docks_available = 0 THEN 1 ELSE 0 END), 1) AS pct_full

FROM workspace.velib_silver.station_status AS r
JOIN workspace.velib_silver.stations       AS s ON r.station_id = s.station_id

-- On garde seulement les données fraîches (signal de moins de 3 h)
WHERE r.last_reported_time >= r.snapshot_time - INTERVAL 3 HOURS

GROUP BY s.station_id, s.name, hour_of_day, day_type;

num_affected_rows,num_inserted_rows


## 3. Vue globale du réseau
Une ligne par relevé : total de vélos sur tout le réseau, et nombre de stations vides ou pleines.

In [0]:
%sql
-- =====================================================================
-- Totaux sur tout le réseau, pour chaque heure de relevé
-- Les totaux comptent seulement les stations connectées (données fraîches)
-- =====================================================================
CREATE OR REPLACE TABLE workspace.velib_gold.network_summary AS
SELECT
  snapshot_time,
  COUNT(*) AS nb_stations,   -- toutes les stations installées

  -- Stations déconnectées : dernier signal il y a plus de 3 h
  SUM(CASE WHEN last_reported_time <  snapshot_time - INTERVAL 3 HOURS THEN 1 ELSE 0 END) AS nb_disconnected,

  -- Totaux calculés seulement sur les stations connectées
  SUM(CASE WHEN last_reported_time >= snapshot_time - INTERVAL 3 HOURS THEN mechanical_bikes ELSE 0 END) AS total_mechanical,
  SUM(CASE WHEN last_reported_time >= snapshot_time - INTERVAL 3 HOURS THEN ebikes           ELSE 0 END) AS total_ebikes,
  SUM(CASE WHEN last_reported_time >= snapshot_time - INTERVAL 3 HOURS THEN bikes_available  ELSE 0 END) AS total_bikes,

  -- Stations connectées ET en service, sans vélo / sans place
  -- (même définition que le statut "vide" / "pleine" de station_latest)
  SUM(CASE WHEN last_reported_time >= snapshot_time - INTERVAL 3 HOURS
            AND (is_renting OR is_returning)
            AND bikes_available = 0 THEN 1 ELSE 0 END) AS nb_empty_stations,
  SUM(CASE WHEN last_reported_time >= snapshot_time - INTERVAL 3 HOURS
            AND (is_renting OR is_returning)
            AND docks_available = 0 THEN 1 ELSE 0 END) AS nb_full_stations

FROM workspace.velib_silver.station_status
GROUP BY snapshot_time
ORDER BY snapshot_time;

num_affected_rows,num_inserted_rows


## 4. Premiers résultats

In [0]:
%sql
-- Vue globale du réseau
SELECT * FROM workspace.velib_gold.network_summary;

snapshot_time,nb_stations,total_mechanical,total_ebikes,total_bikes,nb_empty_stations,nb_full_stations
2026-10-24T18:50:58.000Z,1516,10962,6972,17934,129,78


In [0]:
%sql
-- Top 10 des stations avec le plus de bornes indisponibles
SELECT name, capacity, bikes_available, docks_available, unavailable_docks
FROM workspace.velib_gold.station_latest
ORDER BY unavailable_docks DESC
LIMIT 10;

name,capacity,bikes_available,docks_available,unavailable_docks
Marignan - Champs-Elysées,40,0,0,40
Choiseul - Quatre Septembre,44,5,1,38
Saint-Placide - Cherche Midi,66,37,7,22
Wilson - Landy,40,8,14,18
Arsène Houssaye - Champs-Elysées,52,0,38,14
Invalides - Quai d'Orsay,410,0,400,10
Eugène Reisz - Davout,20,2,8,10
Alibert - Jemmapes,60,7,44,9
Concorde - Place de la Madeleine,409,0,400,9
Gare de l'Est - Chateau Landon,59,6,45,8


num_affected_rows,num_inserted_rows


### Répartition des stations par statut
Combien de stations sont normales, vides, pleines ou hors service au dernier relevé.

In [0]:
%sql
-- Nombre de stations par statut
SELECT status, COUNT(*) AS nb_stations
FROM workspace.velib_gold.station_latest
GROUP BY status
ORDER BY nb_stations DESC;

status,nb_stations
normal,1209
hors service,150
pleine,73
vide,68
déconnectée,16


In [0]:
%sql
-- Les stations "hors service" ont-elles quand même des vélos ?
SELECT
  COUNT(*)                                              AS nb_hors_service,
  SUM(CASE WHEN bikes_available > 0 THEN 1 ELSE 0 END)  AS avec_velos,
  SUM(CASE WHEN docks_available > 0 THEN 1 ELSE 0 END)  AS avec_places,
  ROUND(AVG(unavailable_docks), 1)                       AS moy_bornes_indispo
FROM workspace.velib_gold.station_latest
WHERE status = 'hors service';

nb_hors_service,avec_velos,avec_places,moy_bornes_indispo
164,103,159,0.9


In [0]:
%sql
-- Les stations "hors service" ont-elles arrêté de communiquer ?
-- retard = temps écoulé entre le dernier signal de la station et l'heure du fichier
SELECT
  CASE WHEN is_renting = 1 THEN 'en service' ELSE 'hors service' END AS etat,
  COUNT(*)                                                        AS nb_stations,
  ROUND(AVG((last_updated_epoch - last_reported) / 3600), 1)      AS retard_moyen_heures,
  ROUND(MAX((last_updated_epoch - last_reported) / 3600), 1)      AS retard_max_heures
FROM workspace.velib_bronze.station_status
WHERE is_installed = 1
GROUP BY 1;

etat,nb_stations,retard_moyen_heures,retard_max_heures
en service,1352,0.7,31.9
hors service,164,1000.1,78075.2


### Analyse des stations indisponibles
- 164 stations avaient `is_renting = false`. Le retard moyen de leur dernier signal (≈ 1 000 h) laissait penser à des stations déconnectées.
- En réalité, cette moyenne était faussée par quelques valeurs extrêmes (jusqu'à ≈ 9 ans).
- Avec une règle « pas de signal depuis plus de 3 h », seules **16 stations** sont vraiment **déconnectées** : elles sont exclues des moyennes et des totaux.
- Les **150 autres** communiquent normalement, mais sont **bloquées par l'opérateur** (statut « hors service »).